# SVM MFCC-40
Trains linear SVM multilabel and binary UUV classifiers for normal, M-filtered, and W-filtered MFCC-40 data.

In [ ]:
import sys
from pathlib import Path

UTILS_GITHUB_RAW_BASE_URL = "https://raw.githubusercontent.com/Nabuhodonozzor/uuv-detection/main/utils"
COMMON_UTILS_FILE = "common_utils.py"
MODEL_UTILS_FILE = "svm_utils.py"
MODEL_DIR = "SVM"
common_dirs = [Path.cwd() / "utils", Path.cwd().parent / "utils", Path("/content/utils"), Path("/content/drive/MyDrive/STUDA/src/utils")]
model_dirs = [Path.cwd(), Path.cwd() / MODEL_DIR, Path.cwd().parent / MODEL_DIR, Path("/content") / MODEL_DIR, Path("/content/drive/MyDrive/STUDA/src") / MODEL_DIR]
common_dir = next((directory for directory in common_dirs if (directory / COMMON_UTILS_FILE).exists()), None)
model_dir = next((directory for directory in model_dirs if (directory / MODEL_UTILS_FILE).exists()), None)

if (common_dir is None or model_dir is None) and UTILS_GITHUB_RAW_BASE_URL:
    import urllib.request
    common_dir = Path("/content/utils")
    model_dir = Path("/content") / MODEL_DIR
    common_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{UTILS_GITHUB_RAW_BASE_URL}/common_utils.py", common_dir / COMMON_UTILS_FILE)
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/Nabuhodonozzor/uuv-detection/main/{MODEL_DIR}/{MODEL_UTILS_FILE}", model_dir / MODEL_UTILS_FILE)

if common_dir is None or model_dir is None:
    raise FileNotFoundError("Required common and model utility files were not found. Sync, upload, mount, or set UTILS_GITHUB_RAW_BASE_URL.")

sys.path.insert(0, str(common_dir))
sys.path.insert(0, str(model_dir))
print(f"Using common utilities from: {common_dir}")
print(f"Using SVM utilities from: {model_dir}")


In [ ]:
import pandas as pd
from IPython.display import display
from google.colab import files

from common_utils import configure_kaggle_access, evaluate_models_for_variants, extract_zip, prepare_mfcc_dataset_variants, zip_artifacts
from svm_utils import build_svm_models_for_variants, save_svm_artifacts, train_svm_models_for_variants


In [ ]:
DATASET_KEY = "mfcc40"
DATASET_LABEL = "MFCC-40"
DATASET_SLUG = "pawedyrda/mfcc12"
ARCHIVE_PATH = Path("/content/mfcc12.zip")


In [ ]:
configure_kaggle_access()
!kaggle datasets download -d {DATASET_SLUG} -p /content --force
DATA_PATH = extract_zip(ARCHIVE_PATH, "/content")
print(f"Dataset extracted to: {DATA_PATH}")


In [ ]:
variants = prepare_mfcc_dataset_variants(DATA_PATH)
print("Normal train shape:", variants.normal.train_data.shape)
print("M train shape:", variants.m.train_data.shape)
print("W train shape:", variants.w.train_data.shape)


In [ ]:
multilabel_models = build_svm_models_for_variants(variants, model_type="multilabel")
train_svm_models_for_variants(multilabel_models, variants, model_type="multilabel")

binary_models = build_svm_models_for_variants(variants, model_type="binary")
train_svm_models_for_variants(binary_models, variants, model_type="binary")


In [ ]:
multilabel_results = evaluate_models_for_variants(multilabel_models, variants, "multilabel", DATASET_LABEL)
binary_results = evaluate_models_for_variants(binary_models, variants, "binary", DATASET_LABEL)
comparison_results = pd.concat([
    multilabel_results.assign(task="multilabel"),
    binary_results.assign(task="binary"),
], ignore_index=True)
display(comparison_results[["task", "Model", "precision", "recall", "f1-score", "support"]])


In [ ]:
save_dir = save_svm_artifacts(
    f"/content/saved_artifacts/svm_{DATASET_KEY}", DATASET_KEY, multilabel_models, binary_models,
    multilabel_results, binary_results,
)
comparison_results.to_csv(save_dir / f"svm_comparison_{DATASET_KEY}.csv", index=False)
archive_path = zip_artifacts(save_dir, f"/content/svm_models_and_results_{DATASET_KEY}.zip")
files.download(str(archive_path))
